# Gaussian Shading 官方原始环境复现与 governed import 入口

该 Notebook 服务第二条链路: 官方原始环境复现 + governed import。它不把 legacy Stable Diffusion 结果混入 SD3.5 主表, 只生成补充表所需的官方参考记录、环境报告、命令日志、schema、validation report 和 Google Drive 压缩包。

运行依赖: 不依赖主方法前序结果包, 但必须共享当前 `SLM_WM_PAPER_RUN_NAME` 派生的 prompt split、样本数和目标 FPR。四个 official reference 入口之间互不依赖, 可以在独立 Colab 会话中并行运行; 完整结果包闭合时会从 `external_baseline_official_reference` Drive 子目录读取这些包。

Notebook 通过 `paper_workflow/notebook_utils/notebook_entrypoint.py` 分派到 `paper_experiments/runners/gaussian_shading_official_reference.py`, 只负责 Colab 冷启动、参数注入、远程执行和打包。

## 运行边界

1. 默认样本数由 `SLM_WM_PAPER_RUN_SAMPLE_COUNT` 决定; `pilot_paper` 对应 700 个 prompt, `full_paper` 对应 7000 个 prompt, 用于 fixed-FPR=0.01 共同协议下的官方 legacy reference 复现。
2. Notebook 会在 helper 中按 `external_baseline/source_registry.json` 自动补齐 Gaussian Shading 官方源码快照; 源码快照仍按外部缓存治理, 不进入 git 提交。
3. 该入口默认把 `SLM_WM_GAUSSIAN_SHADING_OFFICIAL_MODEL_ID` 设为 `Manojb/stable-diffusion-2-1-base`, 该路径是 `stabilityai/stable-diffusion-2-1-base` 不可直接访问时的公开镜像回退。
4. 该入口默认开启 `SLM_WM_GAUSSIAN_SHADING_PATCH_MODEL_REPOSITORY_LAYOUT=1`, 会移除官方源码中对 `revision='fp16'` 分支的硬编码, 因为公开镜像把 fp16 相关文件放在 `main` 而不是 `fp16` 分支。
5. 该入口默认开启 `SLM_WM_GAUSSIAN_SHADING_PREPARE_LOCAL_MODEL_REPOSITORY=1`, 会把公开镜像下载到 `/content/gaussian_shading_model_repository/stable_diffusion_2_1_base`, 并把本地 `model_index.json` 中的 `CLIPImageProcessor` 补丁为 legacy diffusers 可读取的 `CLIPFeatureExtractor`。
6. 该入口使用 Python 3.8 和官方 `requirements.txt` 构建严格环境; 依赖安装或导入失败将直接阻断官方参考结果。
7. 若 Colab GPU、legacy wheel 或官方依赖组合不可用, helper 会把失败原因写入 `outputs/gaussian_shading_official_reference/` 并阻断正式归档; 诊断仍保留在当前论文层级输出目录。
8. 若官方原始环境在外部机器完成运行, 可设置 `SLM_WM_GAUSSIAN_SHADING_OFFICIAL_RUN_COMMAND=0`, 并通过 `SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SUMMARY_IMPORT_PATH` 或 `SLM_WM_GAUSSIAN_SHADING_OFFICIAL_LOG_IMPORT_PATH` 导入受治理结果。


In [ ]:
SLM_WM_PAPER_RUN_NAME = "pilot_paper"

import os

from google.colab import drive

drive.mount('/content/drive')
# 修改为 "full_paper" 可切换完整论文运行层级。
os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME


In [ ]:
%pip install -q --upgrade packaging huggingface_hub


In [ ]:
import os
import subprocess
import sys

repository_url = os.environ.get('SLM_WM_REPOSITORY_URL', 'https://github.com/RICHAAARC/SLM-WM.git')
repository_ref = os.environ.get('SLM_WM_REPOSITORY_REF', 'main')
workspace_dir = os.environ.get('SLM_WM_WORKSPACE_DIR', '/content/slm_wm_repository')

if not os.path.exists(workspace_dir):
    subprocess.run(['git', 'clone', repository_url, workspace_dir], check=True)

subprocess.run(['git', 'fetch', '--all'], cwd=workspace_dir, check=True)
subprocess.run(['git', 'checkout', repository_ref], cwd=workspace_dir, check=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)
print({'workspace_dir': workspace_dir, 'repository_ref': repository_ref})

from paper_workflow.colab_utils.paper_run_environment import configure_paper_run_environment
paper_run_environment = configure_paper_run_environment(
    workflow_name="official_reference_gaussian_shading",
)
print(paper_run_environment)

from paper_workflow.colab_utils.dependency_check import build_notebook_dependency_report
dependency_report = build_notebook_dependency_report("official_reference_light")
print(dependency_report)


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get('HF_TOKEN')
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = token_from_secret

if os.environ.get('HF_TOKEN'):
    from huggingface_hub import login
    login(token=os.environ['HF_TOKEN'])
    print('huggingface_login_ready')
else:
    print('HF_TOKEN_not_found')


In [ ]:
from pathlib import Path

source_dir = Path(os.environ['SLM_WM_GAUSSIAN_SHADING_OFFICIAL_SOURCE_DIR'])
requirements_path = source_dir / 'requirements.txt'
entrypoint_path = source_dir / 'run_gaussian_shading.py'
print({
    'source_dir': str(source_dir),
    'source_dir_exists_before_helper': source_dir.exists(),
    'entrypoint_exists_before_helper': entrypoint_path.exists(),
    'requirements_exists_before_helper': requirements_path.exists(),
    'source_prepare_policy': 'helper_will_clone_from_external_baseline_source_registry_when_missing',
})
if requirements_path.exists():
    print(requirements_path.read_text(encoding='utf-8')[:4000])


In [ ]:
import torch

device_report = {
    'cuda_available': torch.cuda.is_available(),
    'device_count': torch.cuda.device_count(),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none',
}
print(device_report)


In [ ]:
import os
from pathlib import Path

from paper_workflow.notebook_utils.notebook_entrypoint import run_workflow

gaussian_shading_official_reference_summary = run_workflow(root='.', workflow_name="official_reference_gaussian_shading")
output_dir = Path('outputs/gaussian_shading_official_reference') / os.environ['SLM_WM_PAPER_RUN_NAME']
legacy_prepare_path = output_dir / 'gaussian_shading_legacy_environment_prepare_result.json'
model_prepare_path = output_dir / 'gaussian_shading_model_repository_prepare_result.json'
for diagnostic_path in (legacy_prepare_path, model_prepare_path):
    if diagnostic_path.exists():
        print(diagnostic_path)
        print(diagnostic_path.read_text(encoding='utf-8')[:6000])
gaussian_shading_official_reference_summary


In [ ]:
from paper_workflow.notebook_utils.notebook_entrypoint import package_workflow_outputs

archive_record = package_workflow_outputs(root='.', workflow_name="official_reference_gaussian_shading")
archive_record.to_dict()


In [ ]:
import os
from pathlib import Path

output_dir = Path('outputs/gaussian_shading_official_reference') / os.environ['SLM_WM_PAPER_RUN_NAME']

for result_path in sorted(output_dir.rglob('*')):
    if result_path.is_file() and result_path.suffix.lower() in {'.json', '.jsonl', '.txt'}:
        print(result_path)
        print(result_path.read_text(encoding='utf-8', errors='ignore')[:4000])

expected_sample_count = int(os.environ['SLM_WM_PAPER_RUN_EXPECTED_SAMPLE_COUNT'])
assert gaussian_shading_official_reference_summary['sample_count'] == expected_sample_count, gaussian_shading_official_reference_summary
if gaussian_shading_official_reference_summary['run_decision'] != 'pass':
    print({
        'gaussian_shading_official_reference_ready': gaussian_shading_official_reference_summary.get('gaussian_shading_official_reference_ready'),
        'unsupported_reason': gaussian_shading_official_reference_summary.get('unsupported_reason'),
        'legacy_environment_ready': gaussian_shading_official_reference_summary.get('legacy_environment_ready'),
        'official_command_return_code': gaussian_shading_official_reference_summary.get('official_command_return_code'),
    })
else:
    assert gaussian_shading_official_reference_summary['reference_import_ready'] is True, gaussian_shading_official_reference_summary
    assert gaussian_shading_official_reference_summary['governed_reference_record_count'] > 0, gaussian_shading_official_reference_summary
